In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-10-27
Last Modified: 2026-10-27
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from core.data import session_pattern
from utils.paths import DATA_DIR
import os


def get_model_params(subj_id, sess_id):
    with np.load(DATA_DIR / subj_id / "hmm_outputs.npz") as f:
        agent_params_obs = f["agent_params_obs"]  # (158, 2)
        hist_predicted_states = f["hist_predicted_states"]  # (158,)
        block_bounds = f["block_bounds"].tolist()

    session_ids_active = session_ids[np.where(subject_ids == subj_id)[0][0]]
    try:
        session_ids_archive = [
            sess_id
            for sess_id in os.listdir(DATA_DIR / "archive" / subj_id)
            if session_pattern.match(sess_id)
        ]
    except FileNotFoundError:
        session_ids_archive = []

    session_ids_15 = np.sort(np.concatenate((session_ids_active, session_ids_archive)))
    assert len(session_ids_15) == 15

    sess_idx_15 = np.where(session_ids_15 == sess_id)[0][0]
    bounds = [0, *block_bounds[:15]]
    bound_idxs = bounds[sess_idx_15 : sess_idx_15 + 1 + 1]

    model_params = agent_params_obs[bound_idxs[0] : bound_idxs[1]]
    strategy_blocks = hist_predicted_states[bound_idxs[0] : bound_idxs[1]]

    return model_params, strategy_blocks


def get_mf_params(trial_data, model_params, strategy_blocks):
    mf_blocks = np.where(strategy_blocks == 0)[0]
    mf_params = model_params[mf_blocks]

    qs = {"L": [], "R": []}
    rpes = []

    for i, (block_idx, (alpha, _)) in enumerate(zip(mf_blocks, mf_params)):
        response_block = trial_data["response"][trial_data["iblock"] == block_idx + 1]
        rewarded_block = trial_data["rewarded"][trial_data["iblock"] == block_idx + 1]

        for j, (choice, reward) in enumerate(zip(response_block, rewarded_block)):
            choice_key = "L" if choice == 1 else ("R" if choice == -1 else None)

            assert choice_key is not None
            choice_key_c = "R" if choice_key == "L" else "L"

            if i == 0 and j == 0:
                q_stale = 0.5
                q_stale_c = 0.5
            else:
                q_stale = qs[choice_key][-1]
                q_stale_c = qs[choice_key_c][-1]

            print(reward, q_stale, reward - q_stale)
            rpe = reward - q_stale
            rpes.append(rpe)

            qs[choice_key].append(q_stale + alpha * rpe)
            qs[choice_key_c].append(q_stale_c)

    rpes = np.array(rpes)
    qs = {k: np.array(v) for k, v in qs.items()}

    return rpes, qs


def get_mb_params(trial_data, model_params, strategy_blocks):
    mb_blocks = np.where(strategy_blocks == 1)[0]
    mb_params = model_params[mb_blocks]

    bs = {"L": [], "R": []}
    spes = []

    for i, (block_idx, (p_switch, p_reward)) in enumerate(zip(mb_blocks, mb_params)):
        response_block = trial_data["response"][trial_data["iblock"] == block_idx + 1]
        rewarded_block = trial_data["rewarded"][trial_data["iblock"] == block_idx + 1]

        for j, (choice, reward) in enumerate(zip(response_block, rewarded_block)):
            choice_key = "L" if choice == 1 else ("R" if choice == -1 else None)

            assert choice_key is not None
            choice_key_c = "R" if choice_key == "L" else "L"

            if i == 0 and j == 0:
                b_stale = 0.5
                b_stale_c = 0.5
            else:
                b_stale = bs[choice_key][-1]
                b_stale_c = bs[choice_key_c][-1]

            spe = reward - b_stale  # reward is fine assuming 100:0 and not 90:10
            spes.append(spe)

            p_reward_cond = p_reward if reward else (1 - p_reward)
            p_reward_cond_c = (1 - p_reward) if reward else p_reward

            b = (
                (1 - p_switch) * p_reward_cond * b_stale
                + (p_switch) * p_reward_cond_c * b_stale_c
            ) / (p_reward * b_stale + (1 - p_reward) * b_stale_c)

            bs[choice_key].append(b)
            bs[choice_key_c].append(1 - b)

    spes = np.array(spes)
    bs = {k: np.array(v) for k, v in bs.items()}

    return spes, bs

In [ ]:
from core.data import default_sessions

model_params, strategy_blocks = get_model_params(
    subj_id="MR82", sess_id=default_sessions["MR82"]
)

In [ ]:
from sg.models import Encoder

encoder = Encoder("MR82", default_sessions["MR82"])
encoder.get_data()

In [ ]:
model_params[np.where(strategy_blocks == 0)]

In [ ]:
rpes, qs = get_mf_params(encoder.trial_data, model_params, strategy_blocks)
spes, bs = get_mb_params(encoder.trial_data, model_params, strategy_blocks)

In [ ]:
model_params[np.where(strategy_blocks == 0)]

In [ ]:
-0.5 + (0.7575 * 0.5)

In [ ]:
plt.figure()
plt.plot(qs["L"])
plt.plot(
    np.array((encoder.trial_data["response"][encoder.trial_data["strategy"] == -1]))
)
plt.plot(
    np.array((encoder.trial_data["rewarded"][encoder.trial_data["strategy"] == -1]))
)
plt.plot(qs["R"])
# plt.plot(spes)
# plt.plot(np.array((encoder.trial_data['bswitch_itrial'][encoder.trial_data['strategy']==1])/25))
# plt.plot(bs['L'])
# # plt.plot(np.array((encoder.trial_data['response'][encoder.trial_data['strategy']==1])))
# plt.plot(np.array((encoder.trial_data['rewarded'][encoder.trial_data['strategy']==1])))
plt.show()

In [ ]:
from sg.models import Encoder, StrategyEncoder
from core.data import default_sessions, subject_ids, session_ids
from core.viz import plot_grouped_boxes
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for subj_id in ["MR82", "MR83", "MR94", "MR95"]:
    encoder = Encoder(
        subj_id,
        default_sessions[subj_id],
        binwidth_ms=100,
        norm=True,
    )
    encoder.verify()

    encoder_mb = StrategyEncoder(
        subj_id,
        default_sessions[subj_id],
        binwidth_ms=100,
        strategy_params="mb",
        add_interaction=True,
        norm=True,
        strategy_filter="mb",
    )
    encoder_mf = StrategyEncoder(
        subj_id,
        default_sessions[subj_id],
        binwidth_ms=100,
        strategy_params="mf",
        add_interaction=True,
        norm=True,
        strategy_filter="mf",
    )
    encoder_mb.verify()
    encoder_mf.verify()

    weights = {strat: {reg: [] for reg in encoder.regions} for strat in ["mb", "mf"]}
    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        e_mb = StrategyEncoder(
            subj_id,
            sess_id,
            binwidth_ms=100,
            strategy_params="mb",
            balance_strategy=False,
            add_interaction=True,
            norm=True,
            strategy_filter="mb",
        )
        e_mf = StrategyEncoder(
            subj_id,
            sess_id,
            binwidth_ms=100,
            strategy_params="mf",
            balance_strategy=False,
            add_interaction=True,
            norm=True,
            strategy_filter="mf",
        )

        try:
            e_mb.fit_encoder()
            e_mf.fit_encoder()
        except ValueError:
            continue

        assert np.all(e_mb.regions == e_mf.regions)
        for reg in e_mb.regions:
            weights["mb"][reg].append(e_mb.encoder_weights[e_mb.reg_idxs[reg]])
            weights["mf"][reg].append(e_mf.encoder_weights[e_mb.reg_idxs[reg]])

    weights = {
        strat: {reg: np.concatenate(weights[strat][reg]) for reg in encoder.regions}
        for strat in ["mb", "mf"]
    }

    models_strat = {"mb": encoder_mb, "mf": encoder_mf}
    tv_idxs = {strat: model.tv_idxs for (strat, model) in models_strat.items()}
    tv_names = {strat: model.tv_names for (strat, model) in models_strat.items()}

    fig, axes = plt.subplots(
        ncols=3, nrows=2, figsize=(8, 7), sharey=True, tight_layout=True
    )

    for i, (strat, weights_strat) in enumerate(weights.items()):
        for j, tv_type in enumerate(["task_vars", "interaction", "strategy_params"]):
            _, ax = plot_grouped_boxes(
                data={
                    regr: {
                        reg: np.abs(weights_strat[reg][:, tv_idxs[strat][regr]])
                        for reg in encoder.regions
                    }
                    for regr in tv_names[strat][tv_type]
                },
                colors=colors_region,
                ylim=[-0.1, 1],
                ax=axes[i][j],
            )

            if i == 0:
                ax.set_title(tv_type)
            if j == 0:
                ax.set_ylabel(rf"$\beta$ distribution, {strat}")

    fig.suptitle(subj_id)

    save_fig(
        fig,
        FIGURES_DIR / "bweight" / "boxplot_comp",
        f"{subj_id}_bweight_comp_rect.png",
    )